# 04 — Resource scaling of the edge- and path-based QUBO formulations

The direct and reduced formulations have now been validated on canonical instances for which the optimum and the relevant routing structure can be checked explicitly. The next question is no longer whether the encodings are correct, but how their logical size and interaction structure evolve as the OEC problem becomes larger.

This notebook therefore studies the resource cost of the two QUBO formulations before introducing QAOA. The analysis is intentionally solver-independent: it measures the binary variables, slack variables and quadratic couplings induced by the mathematical encoding itself, together with the density and degree structure of the corresponding QUBO interaction graph. These quantities determine the number of logical qubits and the number of pairwise cost-Hamiltonian interactions that a gate-based implementation would have to represent, even before compilation to a particular device topology is considered.

In [ ]:
from __future__ import annotations

from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from oec_qaoa.edge_qubo import build_edge_qubo
from oec_qaoa.generators import SyntheticConfig, synthetic_relay_instance
from oec_qaoa.metrics import qubo_resource_metrics
from oec_qaoa.path_qubo import build_path_qubo
from oec_qaoa.path_reduction import candidate_sets

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RESULTS = ROOT / "results" / "scaling"
RESULTS.mkdir(parents=True, exist_ok=True)

## Controlled synthetic family

The scaling study uses a family of source-relay OEC instances rather than extending C1 or C2 by hand. Each instance contains one acquisition satellite, a configurable number of relay satellites and one virtual ground station. Services are generated at the acquisition satellite, can be forwarded to any relay at the initial cycle, stored until the processing window, processed on board or left unprocessed, and finally downlinked before the deadline.

This construction preserves the same time-expanded operations used in the reference model while making the principal dimensions independently adjustable. The parameters varied below are the number of satellites, the number of time cycles, the number of services and the number (K) of retained candidate paths. A separate capacity factor controls how tightly the shared processing and downlink resources are dimensioned around an approximately balanced distribution of the service load.

The resulting family is not intended as a reconstruction of a particular orbital mission. Its role is to isolate the combinatorial consequences of the encoding, after which realistic OEC parameters can be reintroduced without confusing physical assumptions with purely structural scaling effects.

In [ ]:
BASE = {
    "n_satellites": 4,
    "n_time_cycles": 5,
    "n_services": 4,
    "capacity_factor": 1.0,
    "seed": 20260929,
}

example = synthetic_relay_instance(SyntheticConfig(**BASE))
print(example.name)
print(f"Services: {len(example.services)}")
print(f"Graph edges: {len(example.edges)}")

## Resource metrics

For each QUBO, the analysis records

[
N_{\mathrm{semantic}},
\qquad
N_{\mathrm{slack}},
\qquad
N_{\mathrm{logical}}
=
N_{\mathrm{semantic}}+N_{\mathrm{slack}},
]

together with the number (N_{ZZ}) of non-zero quadratic couplings. The interaction density is defined as

[
\rho_{\mathrm{QUBO}}
=
\frac{2N_{ZZ}}
{N_{\mathrm{logical}}(N_{\mathrm{logical}}-1)},
]

for (N_{\mathrm{logical}}>1). The interaction graph also provides the maximum and mean degree and the number of connected components. Finally, the coefficient dynamic range is reported because penalty encodings may increase not only the number of terms but also the spread of the Hamiltonian coefficients.

For the cost Hamiltonian obtained by the usual binary-to-Ising transformation, each non-zero off-diagonal QUBO coupling induces one pairwise (Z_iZ_j) term before hardware-specific compilation. The quantity (N_{ZZ}) is therefore retained as a simple formulation-level proxy for the number of two-qubit cost interactions.

In [ ]:
edge_model = build_edge_qubo(example)
path_model = build_path_qubo(example, candidate_sets(example, k=2))

pd.DataFrame(
    [
        {
            "formulation": "edge",
            **qubo_resource_metrics(edge_model).to_dict(),
        },
        {
            "formulation": "path, K=2",
            **qubo_resource_metrics(path_model).to_dict(),
        },
    ]
)

## One-parameter sweeps

The first set of experiments varies one structural parameter at a time around the common baseline. This separation is useful because the direct encoding reacts differently to the temporal horizon, the number of services and the number of available relays, whereas the reduced formulation is also controlled explicitly by (K).

The initial ranges are deliberately moderate. Their purpose is to reveal the scaling regime and to identify the instance sizes that remain suitable for exact reference calculations and ideal-state QAOA, rather than to force the classical construction to the largest instance that can be generated.

In [ ]:
sweeps = {
    "n_services": [2, 4, 6, 8, 10, 12],
    "n_time_cycles": [3, 4, 5, 6, 8],
    "n_satellites": [3, 4, 5, 6, 8],
}

K_VALUES = [1, 2, 4, 8]
SEEDS = [20260929, 20260930, 20261001]

records = []

for parameter, values in sweeps.items():
    for value in values:
        for seed in SEEDS:
            configuration = dict(BASE)
            configuration[parameter] = value
            configuration["seed"] = seed

            instance = synthetic_relay_instance(
                SyntheticConfig(**configuration)
            )

            edge_qubo = build_edge_qubo(instance)
            edge_metrics = qubo_resource_metrics(edge_qubo)

            records.append(
                {
                    "sweep": parameter,
                    "sweep_value": value,
                    "seed": seed,
                    "formulation": "edge",
                    "K": None,
                    **configuration,
                    **edge_metrics.to_dict(),
                }
            )

            all_paths = candidate_sets(instance)
            max_available = min(
                len(paths)
                for paths in all_paths.values()
            )

            for k in K_VALUES:
                if k > max_available:
                    continue

                paths = {
                    service: candidates[:k]
                    for service, candidates in all_paths.items()
                }
                path_qubo = build_path_qubo(instance, paths)
                path_metrics = qubo_resource_metrics(path_qubo)

                records.append(
                    {
                        "sweep": parameter,
                        "sweep_value": value,
                        "seed": seed,
                        "formulation": "path",
                        "K": k,
                        **configuration,
                        **path_metrics.to_dict(),
                    }
                )

scaling = pd.DataFrame(records)
scaling.head()

In [ ]:
output_csv = RESULTS / "resource_scaling.csv"
scaling.to_csv(output_csv, index=False)
print(output_csv)

## Aggregated trends

The plots below summarize the structural quantities that will be most relevant when the QAOA experiments are designed. Means are taken across the deterministic seeds used for each configuration; the underlying seed-level data remain available in the saved CSV and will be used for uncertainty summaries when the final paper figures are prepared.

At this stage the plots should be interpreted as encoding diagnostics. A reduction in logical variables does not by itself imply an easier variational optimization problem, and a sparser QUBO does not establish an advantage on a particular quantum processor. The measurements instead identify which source of overhead dominates each formulation and which reduced instances are realistic candidates for the next simulation stage.

In [ ]:
def plot_metric(
    frame: pd.DataFrame,
    sweep: str,
    metric: str,
) -> None:
    subset = frame[frame["sweep"] == sweep].copy()
    grouped = (
        subset.groupby(
            ["sweep_value", "formulation", "K"],
            dropna=False,
        )[metric]
        .mean()
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(7.2, 4.5))

    edge = grouped[grouped["formulation"] == "edge"]
    ax.plot(
        edge["sweep_value"],
        edge[metric],
        marker="o",
        label="edge QUBO",
    )

    for k in K_VALUES:
        path = grouped[
            (grouped["formulation"] == "path")
            & (grouped["K"] == k)
        ]
        if path.empty:
            continue
        ax.plot(
            path["sweep_value"],
            path[metric],
            marker="o",
            label=f"path QUBO, K={k}",
        )

    ax.set_xlabel(sweep.replace("_", " "))
    ax.set_ylabel(metric.replace("_", " "))
    ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
plot_metric(scaling, "n_services", "logical_variables")
plot_metric(scaling, "n_services", "quadratic_couplings")
plot_metric(scaling, "n_services", "qubo_density")

In [ ]:
plot_metric(scaling, "n_time_cycles", "logical_variables")
plot_metric(scaling, "n_satellites", "logical_variables")

## Reduction ratios

For configurations in which both formulations have been constructed, it is useful to report the path-to-edge ratios

[
R_N =
\frac{N_{\mathrm{logical}}^{\mathrm{path}}}
     {N_{\mathrm{logical}}^{\mathrm{edge}}},
\qquad
R_{ZZ} =
\frac{N_{ZZ}^{\mathrm{path}}}
     {N_{ZZ}^{\mathrm{edge}}}.
]

Values below one quantify a structural reduction produced by candidate-path preprocessing. These ratios remain descriptive quantities: they do not incorporate the classical cost of generating the paths and they do not predict the variational optimization quality. Both qualifications will be retained when the numerical results are transferred to the manuscript.

In [ ]:
edge_rows = scaling[
    scaling["formulation"] == "edge"
][
    [
        "sweep",
        "sweep_value",
        "seed",
        "logical_variables",
        "quadratic_couplings",
    ]
].rename(
    columns={
        "logical_variables": "edge_logical_variables",
        "quadratic_couplings": "edge_quadratic_couplings",
    }
)

path_rows = scaling[
    scaling["formulation"] == "path"
].copy()

ratios = path_rows.merge(
    edge_rows,
    on=["sweep", "sweep_value", "seed"],
    how="left",
)

ratios["logical_variable_ratio"] = (
    ratios["logical_variables"]
    / ratios["edge_logical_variables"]
)
ratios["coupling_ratio"] = (
    ratios["quadratic_couplings"]
    / ratios["edge_quadratic_couplings"]
)

ratios.to_csv(
    RESULTS / "resource_reduction_ratios.csv",
    index=False,
)

ratios[
    [
        "sweep",
        "sweep_value",
        "K",
        "logical_variable_ratio",
        "coupling_ratio",
    ]
].head()

## Combined diagnostic grid

After the one-parameter sweeps have been inspected, a smaller combined grid can be used to verify that the observed trends are not artifacts of the baseline configuration. The grid below is intentionally kept separate from the primary sweeps, because it increases the number of generated models substantially and is not required for routine regression testing.

In [ ]:
combined_records = []

for n_satellites, n_time_cycles, n_services, seed in product(
    [3, 4, 6],
    [4, 6],
    [2, 4, 8],
    SEEDS,
):
    config = SyntheticConfig(
        n_satellites=n_satellites,
        n_time_cycles=n_time_cycles,
        n_services=n_services,
        capacity_factor=1.0,
        seed=seed,
    )
    instance = synthetic_relay_instance(config)

    edge = build_edge_qubo(instance)
    combined_records.append(
        {
            "formulation": "edge",
            "K": None,
            "n_satellites": n_satellites,
            "n_time_cycles": n_time_cycles,
            "n_services": n_services,
            "seed": seed,
            **qubo_resource_metrics(edge).to_dict(),
        }
    )

    all_paths = candidate_sets(instance)
    max_available = min(len(paths) for paths in all_paths.values())

    for k in (2, 4):
        if k > max_available:
            continue
        paths = {
            service: candidates[:k]
            for service, candidates in all_paths.items()
        }
        model = build_path_qubo(instance, paths)

        combined_records.append(
            {
                "formulation": "path",
                "K": k,
                "n_satellites": n_satellites,
                "n_time_cycles": n_time_cycles,
                "n_services": n_services,
                "seed": seed,
                **qubo_resource_metrics(model).to_dict(),
            }
        )

combined = pd.DataFrame(combined_records)
combined.to_csv(
    RESULTS / "resource_scaling_combined.csv",
    index=False,
)

combined.head()

## Outputs for the next stage

This notebook produces machine-readable tables rather than freezing numerical conclusions in the manuscript. The saved outputs will be reviewed first, together with the resulting plots, to determine which combinations of formulation, (K) and instance size should enter the ideal QAOA study.

The next notebook will therefore not adopt an arbitrary qubit count. It will select exact-optimum-known instances from the scaling regime established here, with particular attention to the difference between the direct and reduced encodings. This keeps the QAOA benchmark connected to the resource analysis and avoids treating a small variational example as evidence for the behavior of the full OEC problem.